In [ ]:
# Se importan las librerías usadas para transformar fuentes operativas.

from pathlib import Path

import pandas as pd
import plotly.express as px

In [ ]:
# Se definen las rutas de fuentes y del producto analítico permanente.

data_directory = Path("../data")
submission_directory = Path("../submission")
submission_directory.mkdir(exist_ok=True)

In [ ]:
# Se cargan las líneas de pedido y se verifican su grano y campos disponibles.

order_lines = pd.read_csv(data_directory / "order_lines.csv")
order_lines.info()
order_lines.shape, order_lines[["order_id", "line_id"]].duplicated().sum()

In [ ]:
# Se cargan clientes y productos para incorporar atributos de análisis.

customers = pd.read_csv(data_directory / "customers.csv")
products = pd.read_csv(data_directory / "products.csv")
customers.shape, products.shape

In [ ]:
# Se integran las fuentes manteniendo una fila por línea de pedido.

sales_analytics = (
    order_lines.merge(customers, on="customer_id", validate="many_to_one")
    .merge(products, on="product_id", validate="many_to_one")
)
sales_analytics["order_date"] = pd.to_datetime(sales_analytics["order_date"])
sales_analytics["gross_sales"] = sales_analytics["quantity"] * sales_analytics["unit_price"]
sales_analytics["discount_amount"] = sales_analytics["gross_sales"] * sales_analytics["discount_pct"]
sales_analytics["net_sales"] = sales_analytics["gross_sales"] - sales_analytics["discount_amount"]
sales_analytics.shape

In [ ]:
# Se verifica que la transformación preserve líneas, importes y relaciones válidas.

assert len(sales_analytics) == len(order_lines)
assert sales_analytics[["customer_id", "product_id", "region", "category"]].notna().all().all()
assert sales_analytics["net_sales"].sum() == (sales_analytics["gross_sales"] - sales_analytics["discount_amount"]).sum()
sales_analytics[["order_id", "line_id", "region", "category", "net_sales"]].head()

In [ ]:
# Se publica una tabla analítica con una fila por línea de pedido.

sales_analytics.to_csv(submission_directory / "sales_analytics.csv", index=False)
print("Grano publicado: una fila por línea de pedido.")

In [ ]:
# ¿Cómo cambian las ventas netas mensuales por categoría después de integrar las fuentes?

sales_analytics["month"] = sales_analytics["order_date"].dt.to_period("M").dt.to_timestamp()
monthly_category = (
    sales_analytics.groupby(["month", "category"], as_index=False)["net_sales"]
    .sum()
    .sort_values(["month", "category"])
)
monthly_category

In [ ]:
# Se visualizan las ventas netas por categoría sin ocultar la dimensión temporal.

fig = px.line(
    monthly_category, x="month", y="net_sales", color="category", markers=True,
    title="Ventas netas mensuales por categoría",
    labels={"month": "Mes", "net_sales": "Ventas netas", "category": "Categoría"},
)
fig.update_layout(template="plotly_white")
fig.show()